# 02 - Preprocessing
### Phishing URL Detector — PhiUSIIL Dataset

Goal: prepare a clean, model-ready feature matrix — drop identifier/raw-text columns, split into train/test sets, and scale numeric features where needed.

**Note on disk usage:** train/test CSVs (especially the scaled version) are large (100MB+) and would bloat both disk and Git. So this notebook does NOT save train/test splits to disk — everything after loading stays in memory for this session. Only the small, reusable artifacts (`scaler.pkl`, `feature_columns.pkl`) are saved, since the live app needs those later. The next notebook (`03_baseline_models.ipynb`) repeats these same steps in-memory before training, so each notebook is self-contained and safely re-runnable.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
import os

df = pd.read_csv("../data/raw/PhiUSIIL_Phishing_URL_Dataset.csv")
print("Shape:", df.shape)
df.head()

## 1. Clean the Data

Same cleaning steps as `01_eda.ipynb`: drop duplicate URLs and identifier/raw-text columns.
- `FILENAME` — random file identifier, no signal
- `URL`, `Domain` — raw text (lexical signal already captured in engineered numeric columns)
- `TLD` — raw text (numeric proxies `TLDLength`, `TLDLegitimateProb` already exist)
- `Title` — raw scraped text, high cardinality, not usable as-is

In [ ]:
df = df.drop_duplicates(subset='URL').reset_index(drop=True)

cols_to_drop = ['FILENAME', 'URL', 'Domain', 'TLD', 'Title']
cols_to_drop = [c for c in cols_to_drop if c in df.columns]

df_model = df.drop(columns=cols_to_drop)
print("Columns dropped:", cols_to_drop)
print("Remaining shape:", df_model.shape)
df_model.head()

## 2. Sanity Checks Before Splitting

In [ ]:
print("Missing values:", df_model.isnull().sum().sum())
print("Non-numeric columns:", df_model.select_dtypes(exclude=[np.number]).columns.tolist())
print("Final feature count (excluding label):", df_model.shape[1] - 1)

## 3. Train / Test Split

Stratified split on `label` to preserve the class ratio (~57% legitimate / 43% phishing) in both sets.
`random_state=42` is fixed so this split is reproducible — the next notebook re-runs this exact split rather than reading a saved file.

In [ ]:
X = df_model.drop(columns=['label'])
y = df_model['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train label distribution:\n", y_train.value_counts(normalize=True))
print("Test label distribution:\n", y_test.value_counts(normalize=True))

## 4. Feature Scaling

Tree-based models (Random Forest, XGBoost) don't strictly need scaling, but Logistic Regression does.
We fit the scaler on the training set only, then apply it to both sets, to avoid data leakage.

The scaled arrays stay in memory only (not saved to disk) — the fitted `scaler` object is what actually gets reused later, not the scaled data itself.

In [ ]:
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("Scaling complete. Example means (train, should be ~0):")
X_train_scaled.mean().head()

## 5. Save Only the Small, Reusable Artifacts

We save just the fitted `scaler` and the exact list/order of feature columns.
These are small files (a few KB) and are exactly what the live `/scan` endpoint will need later to transform a brand-new URL's features the same way.

No train/test CSVs are written to disk here.

In [ ]:
os.makedirs("../../backend/app/ml/saved_models", exist_ok=True)

joblib.dump(scaler, "../../backend/app/ml/saved_models/scaler.pkl")
joblib.dump(list(X_train.columns), "../../backend/app/ml/saved_models/feature_columns.pkl")

print("Saved: scaler.pkl, feature_columns.pkl -> backend/app/ml/saved_models/")
print("(X_train, X_test, y_train, y_test, and the scaled versions remain in memory only for this session)")

## Summary

- Final feature count, train/test sizes, and class balance are confirmed above.
- Only `scaler.pkl` and `feature_columns.pkl` are persisted to disk — no large CSVs.
- Next notebook (`03_baseline_models.ipynb`) will repeat this same load -> clean -> split -> scale sequence in-memory, then train Logistic Regression, Random Forest, and XGBoost and compare results.